# Baseline: Domain Heuristic

This notebook implements a simple rule-based baseline (no ML). The rule: if `grid <= 10` predict `top-10`, otherwise predict `not top-10`. We evaluate this on a validation set (season 2023) and report accuracy, reflections, and a statement that any model must beat this baseline.

In [3]:
import pandas as pd

# Load cleaned results (file is relative to this notebook's folder)
df = pd.read_csv('data/processed/results_2022_2024_clean.csv')

# Inspect basic columns and types
print('Columns:', list(df.columns))
print('Seasons present:', sorted(df['season'].unique()))

Columns: ['season', 'round', 'race', 'date', 'circuit', 'driverId', 'driver', 'constructor', 'position', 'positionText', 'grid', 'laps', 'status', 'points', 'top10', 'finished']
Seasons present: [2022, 2023, 2024]


In [8]:
# Split: use 2023 as validation (pre-race baseline evaluation)
val = df[df['season'] == 2023].copy()

# Ensure `grid` is numeric and drop rows without a valid grid (pre-race feature)
val['grid'] = pd.to_numeric(val['grid'], errors='coerce')
val = val.dropna(subset=['grid'])

# Apply heuristic: predict top-10 if qualifying grid <= 10
val['pred_top10'] = val['grid'] <= 10

# Ground truth: `top10` column (already boolean-like)
# Compute accuracy on validation set
accuracy = (val['pred_top10'] == val['top10']).mean()

# Also compute baseline that always predicts 'top-10' for comparison
always_top10_acc = val['top10'].mean()

# Report results
print(f'Validation rows: {len(val):,}')
print(f'Heuristic (grid<=10) accuracy on validation: {accuracy:.4f}')
print(f"Always-predict-top10 accuracy on validation: {always_top10_acc:.4f} (fraction of actual top-10)")

# Show a small confusion-style summary
confusion = pd.crosstab(val['top10'], val['pred_top10'], rownames=['actual_top10'], colnames=['pred_top10'])
print('\nConfusion table (rows=actual, cols=pred):')
print(confusion)

Validation rows: 100
Heuristic (grid<=10) accuracy on validation: 0.7400
Always-predict-top10 accuracy on validation: 0.5000 (fraction of actual top-10)

Confusion table (rows=actual, cols=pred):
pred_top10    False  True 
actual_top10              
False            37     13
True             13     37


**Reflection on accuracy**

Is this accuracy good enough to make decisions with? What could accuracy be hiding?

Our heuristic reached **74% accuracy** on **100 validation rows**. Interpreted directly, that means **74 correct predictions** and **26 mistakes**.

Compared to a naive baseline that always predicts `top-10` (**50% accuracy**), the rule adds **+24 percentage points** of signal. So it is clearly better than guessing based only on class frequency.

From the confusion table:
- Correct non-top-10 predictions: **37**
- Correct top-10 predictions: **37**
- Missed top-10 (predicted non-top-10 but actually top-10): **13**
- False top-10 alarms (predicted top-10 but actually non-top-10): **13**

This pattern is balanced, but 26% error can still be costly depending on the decision context. For example, if a wrong top-10 call is expensive, accuracy alone may look better than real decision quality.

If 50% of drivers are actually top-10, a model that always predicts `top-10` would still score **50%** while being useless. That is why we should also check precision/recall by class, not just one accuracy number.

Any model we build in Lab 2 must beat this number. If it doesn't, the model adds no value.